# Notebook 4: Agentic RAG Introduction

In this notebook you will learn how to classify a user search and have specialize agents called depending on the user's question. This notebook does not use the Azure AI Search index to create a full solution for you - you will get to do that in the hands on exercise.

## Learning Objectives
- Learn about the WorkflowBuilder and conditional routing using switch-case logic
- Implement a classifier to recognize the user's intent and route to a specialized agent
- Implement agents to respond to specific search types


### Setup the Module Imports

In [ ]:
import os

from collections.abc import AsyncIterable
from typing import Annotated, cast

from agent_framework import (
    Default,
    Case,
    WorkflowBuilder,
    tool
)
from agent_framework.openai import OpenAIChatClient
from agent_framework.orchestrations import HandoffAgentUserRequest, HandoffBuilder
from azure.identity import DefaultAzureCredential

Define AI functions you'll use for testing the specific search type in this notebook.

> NOTE: these are where the actual implementation goes for doing specific search types in the hands on notebook

In [ ]:
@tool(approval_mode="never_require")
def yes_or_no_search(user_question: Annotated[str, "User question to answer with yes or no"]) -> str:
    """Answers yes or no questions."""
    return f"Question: {user_question}\nAnswer: NO"

@tool(approval_mode="never_require")
def count_search(user_question: Annotated[str, "User question requiring counting items"]) -> str:
    """Answers questinos that required counting."""
    return f"Question: {user_question}\nAnswer: 42"

@tool(approval_mode="never_require")
def semantic_search(user_question: Annotated[str, "User question requiring semantic search"]) -> str:
    """Answers simple question that requires semantic search."""
    return f"Question: {user_question}\nAnswer: They all the gray or metallic in color."

Create a prompt for the classifier to be able to determine how to route the user's question.

In [ ]:
CLASSIFIER_AGENT_INSTRUCTIONS = """
You are a query classification system for an IT support ticket database.
   Classify the incoming user question into exactly one category and return
   a JSON object with "category" and "reasoning" fields.

   ## Database Schema
   The database contains IT support tickets with these fields:
   - Id: unique identifier
   - Create_Date: date the ticket was created
   - Subject: ticket subject
   - Body: ticket question/description
   - Answer: ticket response/solution
   - Type: ticket type (values: "Incident", "Request", "Problem", "Change")
   - Queue: department name (values: "Human Resources", "IT", "Finance", "Operations", "Sales", "Marketing", "Engineering", "Support")
   - Priority: priority level (values: "high", "medium", "low")
   - Language: ticket language
   - Business_Type: business category
   - Tags: categorization tags

   **IMPORTANT**: When "and" combines field values (Type, Queue, Priority), these are FILTERS for counting/searching, NOT separate items to compare.

   ## Categories (use these exact values):

   **difference**: Questions with negation/exclusion ("not", "don't", "without", "excluding").
      - "Which Dell XPS Issue does not mention Windows?" -> difference
      - "Find tickets without high priority" -> difference

   **intersection**: Questions combining multiple SEARCH TOPICS with "and", "both", "that also".
      - "What issues are for Dell XPS laptops and the user tried Win + Ctrl + Shift + B?" -> intersection
      - NOT when "and" combines field filters like Priority/Queue/Type.

   **multi_hop**: Questions asking for a different attribute than what's searched (find X, extract Y).
      - "What department had consultants with Login Issues?" -> multi_hop
      - "Which queue handles password reset requests?" -> multi_hop

   **comparative**: Questions comparing multiple items ("more", "less", "vs", "or").
      - "Do we have more issues with MacBook Air or Dell XPS?" -> comparative

   **yes_no**: Explicit yes/no questions expecting a boolean answer.
      - "Are there any issues for Dell XPS laptops?" -> yes_no

   **count**: Counting questions ("how many", "count", "total", "number of").
      - "How many Incidents for Human Resources and low priority?" -> count (all filters!)

   **semantic_search**: General queries about issues, solutions, how-to (everything else).
      - "What problems are there with Surface devices?" -> semantic_search

   ## Classification Priority
   1. COUNTING first: "how many", "count", "total" -> count
   2. NEGATION: "not", "don't", "without" -> difference
   3. COMPARISON: "more", "less", "vs", "or" comparing items -> comparative
   4. INTERSECTION: multiple search topics with "and" (NOT field filters) -> intersection
   5. MULTI-HOP: "What [FIELD] had [CONDITION]" -> multi_hop
   6. YES/NO: explicit boolean questions -> yes_no
   7. Everything else -> semantic_search

   ## Key Rules
   - Field values (Priority, Queue, Type) are FILTERS, not search topics
   - "How many X and Y and Z?" = count (filters). "What X and Y?" = intersection (topics)
   - "Which X does not mention Y?" = difference, NOT count
"""

### Create the Agents

Define a factory function that creates all four agents from a single `OpenAIChatClient`:

- **Classifier agent** — the central dispatcher. It receives every user message first and decides which specialist to hand off to. Uses the `CLASSIFIER_AGENT_INSTRUCTIONS` prompt to classify the query
- **Specialist agents** (`yes_no_agent`, `count_agent`, `semantic_search_agent`) — each is created with `as_agent()`, given a focused system instruction, and wired to its corresponding `@tool`-decorated search function. Each specialist is also instructed to hand off back to the classifier when done

All agents set `require_per_service_call_history_persistence=True` so conversation history is maintained across handoffs.

In [ ]:
def create_agents(chat_client: OpenAIChatClient) -> tuple[Agent, Agent, Agent, Agent]:
    """Create and configure the classifier and search specialist agents.

    The classifier agent is responsible for:
    - Receiving all user input first
    - Deciding whether to handle the request directly or hand off to a search specialist
    - Signaling handoff by calling one of the explicit handoff tools exposed to it

    Search specialist agents are invoked only when the classifier agent explicitly hands off to them.
    After a specialist responds, control returns to the classifier agent, which then prompts
    the user for their next message.

    Returns:
        Tuple of (classifier_agent, yes_no_agent, count_agent, semantic_search_agent)
    """
    # Classifier agent: Acts as the search planner and dispatcher
    classifier_agent = chat_client.as_agent(
        instructions=CLASSIFIER_AGENT_INSTRUCTIONS,
        name="classifier_agent",
        default_options={"response_format": ClassifyResult},
        require_per_service_call_history_persistence=True,
    )

    # Yes/No search specialist: Handles yes/no questions
    yes_no_agent = chat_client.as_agent(
        instructions="You handle yes/no questions. When done, hand off back to the classifier_agent.",
        name="yes_no_agent",
        tools=[yes_or_no_search],
        require_per_service_call_history_persistence=True,
    )

    # Count search specialist: Handles counting questions
    count_agent = chat_client.as_agent(
        instructions="You handle questions that require counting items. When done, hand off back to the classifier_agent.",
        name="count_agent",
        tools=[count_search],
        require_per_service_call_history_persistence=True,
    )

    # Semantic search specialist: Handles semantic search questions
    semantic_search_agent = chat_client.as_agent(
        instructions="You handle questions that require semantic search. When done, hand off back to the classifier_agent.",
        name="semantic_search_agent",
        tools=[semantic_search],
        require_per_service_call_history_persistence=True,
    )

    return classifier_agent, yes_no_agent, count_agent, semantic_search_agent

### Utility: Print Agent User Requests

When a handoff workflow pauses to request user input (i.e., the agent's response didn't trigger a handoff), this helper prints the agent's messages so the user can see what the agent said before being prompted for more input.

In [ ]:
def _print_handoff_agent_user_request(response: AgentResponse) -> None:
    """Display the agent's response messages when requesting user input.

    This will happen when an agent generates a response that doesn't trigger
    a handoff, i.e., the agent is asking the user for more information.

    Args:
        response: The AgentResponse from the agent requesting user input
    """
    if not response.messages:
        raise RuntimeError("Cannot print agent responses: response has no messages.")

    print("\n[Agent is requesting your input...]")

    # Print agent responses
    for message in response.messages:
        if not message.text:
            # Skip messages without text (e.g., tool calls)
            continue
        speaker = message.author_name or message.role
        print(f"- {speaker}: {message.text}")

### Utility: Handle Workflow Events

The handoff workflow emits a stream of `WorkflowEvent` objects as it runs. This handler processes each event type:

- **`handoff_sent`** — a handoff was initiated from one agent to another (e.g., classifier to count_agent)
- **`status`** — the workflow state changed (e.g., `IDLE`, `IDLE_WITH_PENDING_REQUESTS`)
- **`output`** — an agent produced a response; we print the text from each message
- **`request_info`** — the workflow is pausing to ask for user input; these are collected and returned so we can feed in the next response

In [ ]:
def _handle_events(events: list[WorkflowEvent]) -> list[WorkflowEvent[HandoffAgentUserRequest]]:
    """Process workflow events and extract any pending user input requests.

    This function inspects each event type and:
    - Prints workflow status changes (IDLE, IDLE_WITH_PENDING_REQUESTS, etc.)
    - Displays final conversation snapshots when workflow completes
    - Prints user input request prompts
    - Collects all request_info events for response handling

    Args:
        events: List of WorkflowEvent to process

    Returns:
        List of WorkflowEvent[HandoffAgentUserRequest] representing pending user input requests
    """
    requests: list[WorkflowEvent[HandoffAgentUserRequest]] = []

    for event in events:
        if event.type == "handoff_sent":
            # handoff_sent event: Indicates a handoff has been initiated
            print(f"\n[Handoff from {event.data.source} to {event.data.target} initiated.]")
        elif event.type == "status" and event.state in {
            WorkflowRunState.IDLE,
            WorkflowRunState.IDLE_WITH_PENDING_REQUESTS,
        }:
            # Status event: Indicates workflow state changes
            print(f"\n[Workflow Status] {event.state}")
        elif event.type == "output":
            # Output event: Contains contents generated by the workflow
            data = event.data
            if isinstance(data, AgentResponse):
                for message in data.messages:
                    if not message.text:
                        # Skip messages without text (e.g., tool calls)
                        continue
                    speaker = message.author_name or message.role
                    print(f"- {speaker}: {message.text}")
            elif event.type == "output":
                # The output of the handoff workflow is a collection of chat messages from all participants
                conversation = cast(list[Message], event.data)
                if isinstance(conversation, list):
                    print("\n=== Final Conversation Snapshot ===")
                    for message in conversation:
                        speaker = message.author_name or message.role
                        print(f"- {speaker}: {message.text or [content.type for content in message.contents]}")
                    print("===================================")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            _print_handoff_agent_user_request(event.data.agent_response)
            requests.append(cast(WorkflowEvent[HandoffAgentUserRequest], event))

    return requests

### Wire Everything Together

Now all the code is defined, you get to use it. The cell below:

1. Loads environment variables from `.env` via `dotenv`
2. Creates an `OpenAIChatClient` using `DefaultAzureCredential` for the configured deployment
3. Calls `create_agents` to build the classifier and three specialist agents

In [ ]:
import dotenv
dotenv.load_dotenv()

# Initialize the Azure OpenAI chat client
chat_client = OpenAIChatClient(
    model=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    credential=DefaultAzureCredential()
)

# Create all agents: classifier + specialists
classifier, yes_no, count, semantic_search = create_agents(chat_client)

### Build the Workflow with Handoff Routing

Use the [HandoffBuilder](https://learn.microsoft.com/en-us/python/api/agent-framework-core/agent_framework.handoffbuilder?view=agent-framework-python-latest) to wire the agents into a handoff workflow:

1. **Participants** — all four agents are registered so the builder knows about them
2. **Termination condition** — the workflow ends when the last message starts with `"SEARCH_COMPLETE"`
3. **`with_start_agent`** — sets the classifier as the entry point for every new conversation
4. **`add_handoff`** — defines which agents can hand off to which: the classifier can hand off to any specialist, and each specialist can hand off back to the classifier
5. **`with_autonomous_mode`** — enables the workflow to run handoffs automatically without waiting for user confirmation at each step

In [ ]:
workflow = (
    HandoffBuilder(
        name="agentic_rag_workflow",
        participants=[classifier, yes_no, count, semantic_search],
        termination_condition=lambda conversation: (
            len(conversation) > 0 and conversation[-1].text.strip().startswith("SEARCH_COMPLETE")
        ),
    )
    .with_start_agent(classifier)
    .add_handoff(classifier, [yes_no])
    .add_handoff(classifier, [count])
    .add_handoff(classifier, [semantic_search])
    .add_handoff(yes_no, [classifier])
    .add_handoff(count, [classifier])
    .add_handoff(semantic_search, [classifier])
    .with_autonomous_mode()
    .build()
)

For demo purposes create a list of some sample questions.

In [ ]:
scripted_responses = [
    "How many tickets were logged and Incidents for Human Resources and low priority?", # count search
    "Are there any issues logged for Dell XPS laptops?" # yes/no search
]

### Run the Workflow

The `run_workflow` function below drives the handoff loop:

- **`drain_events`** — a small helper that collects all events from the workflow's async stream into a list so we can process them synchronously
- **`workflow.run(initial_message)`** — kicks off the workflow with the first user question, returning an async stream of `WorkflowEvent` objects
- **Request/response loop** — after each cycle, `_handle_events` returns any pending `request_info` events. For each one, we pop the next scripted question and send it as a `HandoffAgentUserRequest.create_response()`. When scripted responses run out, we send `HandoffAgentUserRequest.terminate()` to end the workflow

In [ ]:
async def drain_events(stream: AsyncIterable[WorkflowEvent]) -> list[WorkflowEvent]:
    """
    Collect all events from an async stream into a list.
    
    This helper drains the workflow's event stream so we can process events
    synchronously after each workflow step completes.
    
    Args:
        stream: Async iterable of WorkflowEvent
        
    Returns:
        List of all events from the stream
    """
    return [event async for event in stream]

async def run_workflow():
    initial_message = "What problems are there with Surface devices?" # semantic search
    print(f"- User: {initial_message}")

    events = await drain_events(workflow.run(initial_message, stream=True))
    pending_requests = _handle_events(events)

    #workflow_result = workflow.run(initial_message, stream=True)
    #pending_requests = _handle_events([event async for event in workflow_result])
    # Process the request/response cycle
    # The workflow will continue requesting input until:
    # 1. The termination condition is met, OR
    # 2. We run out of scripted responses
    while pending_requests:
        if not scripted_responses:
            # No more scripted responses; terminate the workflow
            responses = {req.request_id: HandoffAgentUserRequest.terminate() for req in pending_requests}
        else:
            # Get the next scripted response
            user_response = scripted_responses.pop(0)
            print(f"\n- User: {user_response}")
            # Send response(s) to all pending requests
            # In this demo, there's typically one request per cycle, but the API supports multiple
            responses = {
                req.request_id: HandoffAgentUserRequest.create_response(user_response) for req in pending_requests
            }
        # Send responses and get new events
        # We use run(responses=...) to get events from the workflow, allowing us to
        # display agent responses and handle new requests as they arrive
        events = await workflow.run(responses=responses)
        pending_requests = _handle_events(events)


In [ ]:
await run_workflow()

As I've pointed out in the other notebooks, the question "How many tickets were logged and Incidents for Human Resources and low priority?" is a hard one to get the system to answer - and this time the classifier correctly identified it as something the count_agent should take care of. That is a start!

Next is the hands on exercise where you get to turn these lessons from the notebooks into an Agentic RAG solution for the IT support search index we've been using.